## 使用@tool装饰器来调用工具

### 调用工具示例代码

In [ ]:
from langchain.messages import HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from rich import print as rprint
from langchain_core.tools import tool

## 定义工具，需要使用@tool装饰器，同时作为工具的函数需要有docstring
@tool
def get_weather(city: str="北京"):
    """ 
    查询指定城市天气信息

    Args:
        city : 具体的城市，如:北京等等

    Returns:
        返回城市的天气     
    """

    return f"{city}天气晴朗，万里无云"
model = ChatOpenAI(
    model="qwen3-vl:latest",
    api_key="sk1234567",
    base_url="http://localhost:11434/v1"
)

tmodel = model.bind_tools([get_weather])
msgs = [
    HumanMessage("北京的天气怎么样？"),
]

resp = model.invoke(msgs)
tool_calls = resp.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        tresp = ToolMessage(
            content=get_weather.invoke(**tool_call["args"]),
            tool_call_id=tool_call["id"],
            name=tool_call["name"]
        )

        msgs.append(tresp)
print("========================messages===============================")
for msg in msgs:
    msg.pretty_print()
print("========================messages===============================")
res = model.invoke(msgs)
rprint(res)

========================messages===============================
================================ Human Message =================================

北京的天气怎么样？
========================messages===============================


AIMessage(
    content='你好！关于北京的天气，由于我无法实时获取最新数据，**但可以为你提供一个通用的查询方式和当前常见情况的说
明**：\n\n---\n\n### 一、当前天气（假设今天）：\n- **天气状况**：  \n  根据近期北京气候，通常 
**白天晴朗/多云，夜间偶尔有雷阵雨**（尤其夏季）。若正值秋季，可能 **蓝天白云、微风凉爽**；冬季则 
**晴冷干燥**，偶有雾霾（需关注空气质量）。\n\n- **温度范围**：  \n  - 夏季：**28℃~35℃**（高温炎热）  \n  - 
秋季：**18℃~28℃**（舒适宜人）  \n  - 冬季：**-5℃~10℃**（寒冷干燥）  \n\n- **体感温度**：  \n  炎热时 
**30℃+**（需防暑），干燥时 **体感更低**（尤其北风天）。\n\n---\n\n### 二、重要提示：\n1. **实时数据查询**：  \n   
**建议你打开手机天气APP**（如“中国天气网”“墨迹天气”），输入“北京”即可看到 **实时温度、降水概率、风速** 等详细数据。
\n   📈 *小技巧：搜索“北京天气”+“实时”关键词，结果更精准。*\n\n2. **特殊天气预警**：  \n   - 
**雾霾天**：空气质量差，建议减少外出，戴口罩。  \n   - **雷雨季**（4-9月）：  \n     ⛈️ 
**注意防雷、防洪**，避免在户外停留，驾车需警惕积水。  \n   - 
**暴雨/台风**：如遇强降雨，提前查看交通管制信息。\n\n---\n\n### 三、出行建议：\n- **晴天**：  \n  → 
穿浅色衣物（反射阳光）+ 草帽/遮阳伞（防晒）  \n- **雷雨天**：  \n  → 携带折叠伞，穿防水鞋，避免涉水  \n- **冬季**：
\n  → 厚外套+围巾，保暖防风  \n\n---\n\n### 四、权威预报渠道：\n| 平台 | 优势 | 操作建议 
|\n|------|------|----------|\n| **中国天气网** | 官方精准预报 | 打开APP → 搜索“北京” → 查看实时数据 |\n| 
**AccuWeather** | 全球覆盖+预警 | 搜索“Beijing”查看逐小时天气 |\n| **北京气象局** | 精准本地信息 | 
关注公众号“北京天气”获取实时提醒 
|\n\n---\n\n如果需要更具体的建议（比如**今天**的天气细节），请告诉我你的**出行时间**或**关注点**（如是否带孩子出游
、是否需要穿衣指南），我可以帮你进一步分析！🌞☔️',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 929,
            'prompt_tokens': 15,
            'total_tokens': 944,
            'completion_tokens_details': None,
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 14}
        },
        'model_provider': 'openai',
        'model_name': 'qwen3-vl:latest',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-327',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a0b731-5c49-7221-a60c-9c7bc71d2940-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 15,
        'output_tokens': 929,
        'total_tokens': 944,
        'input_token_details': {'cache_read': 14},
        'output_token_details': {}
    }
)

#### 思考， 为什么本地工具调用不了，网络工具却可以使用？

## 使用Pydantic模型定义
### 1.pydantic类型的定义

In [7]:
from pydantic import BaseModel,Field
from langchain_core.tools import tool

#定义一个类继承之BaseModel
class WeatherInput(BaseModel):
    city:str = Field(
        description="具体的城市",
        default="北京"
    )

## 定义工具，需要使用@tool装饰器，同时作为工具的函数需要有docstring
@tool(args_schema=WeatherInput)
def get_weather(city: str="北京"):
    """ 查询指定城市天气信息"""

    return f"{city}天气晴朗，万里无云"

In [8]:
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询指定城市天气信息',
        'parameters': {
            'properties': {'city': {'default': '北京', 'description': '具体的城市', 'type': 'string'}},
            'type': 'object'
        }
    }
}